# Hemisphere kinematics from the mixer

Kinematic distributions of the per-event **hemispheres** written by
`run3-mj-mixer` (Steps 1–2: thrust-axis split + hemisphere characterization).
Inputs are `mixed_*.root` files; every plot overlays the four hemisphere
jet-multiplicity classes (**1, 2, 3, 4 jets** — the possible sides of an
exactly-5-jet event).

The variables include the matching set of the hemisphere-mixing method
(arXiv:1712.02538, arXiv:2403.20241): hemisphere invariant mass, longitudinal
momentum, $p_T$ parallel / perpendicular to the thrust axis, jet multiplicity —
plus the partner-hemisphere $\eta$ used by our nearest-neighbor scheme.


In [ ]:
import numpy as np
import awkward as ak
import uproot
import matplotlib.pyplot as plt


In [ ]:
# Mixed files from run3-mj-mixer -- EDIT ME (2-4 local or root:// paths,
# relative paths are resolved from notebooks/).
FILES = [
    "../../mixed_slimmed_EXAMPLE_1.root",
    "../../mixed_slimmed_EXAMPLE_2.root",
]
TREE = "events"

# True: area-normalise every curve (shape-only comparison).
# False: stack-free weighted counts (lumi * xs_pb / n_original per hemisphere).
DENSITY = True

NBINS = 60


## Load

Reads the `Hemisphere_*` branches (2 hemispheres/event) plus the event-level
`thrust` and `xs_weight`, and flattens to one entry per hemisphere. The
`mixer_cutflow` per file shows the exactly-5-jet selection.


In [ ]:
BRANCHES = [
    "Hemisphere_pt", "Hemisphere_eta", "Hemisphere_phi", "Hemisphere_mass",
    "Hemisphere_energy", "Hemisphere_pz", "Hemisphere_pt_par",
    "Hemisphere_pt_perp", "Hemisphere_partner_eta", "Hemisphere_n_jets",
    "Hemisphere_side", "Hemisphere_weight",
    "thrust", "xs_weight",
]

for path in FILES:
    with uproot.open(path) as f:
        n = f[TREE].num_entries if TREE in f else 0
        cf = f["mixer_cutflow"].values() if "mixer_cutflow" in f else None
        cut = f"  mixer_cutflow: {cf[0]:.0f} read -> {cf[1]:.0f} kept" if cf is not None else ""
        print(f"{path.split('/')[-1]:<70s} {n:>9,} events{cut}")

events = uproot.concatenate([f"{p}:{TREE}" for p in FILES], BRANCHES, library="ak")

# One flat entry per hemisphere (2 per event, index 0 = +n_T side).
hemi = {k.replace("Hemisphere_", ""): ak.to_numpy(ak.flatten(events[k]))
        for k in BRANCHES if k.startswith("Hemisphere_")}
njets = hemi["n_jets"].astype(int)
w = hemi["weight"]

# Derived per-hemisphere quantities.
hemi["abs_pt_par"] = np.abs(hemi["pt_par"])            # magnitude along n_T
hemi["d_eta_partner"] = hemi["eta"] - hemi["partner_eta"]

# Event-level: (n_events, 2) regular views + thrust.
nj2 = ak.to_numpy(ak.fill_none(ak.pad_none(events["Hemisphere_n_jets"], 2, clip=True), 0)).astype(int)
split_min = nj2.min(axis=1)                            # 2 -> 2+3 split, 1 -> 1+4, 0 -> 0+5
thrust = ak.to_numpy(events["thrust"])
xs_w = ak.to_numpy(events["xs_weight"])

# Population by hemisphere jet multiplicity (weighted fractions used in legends).
FRAC = {n: w[njets == n].sum() / w.sum() for n in np.unique(njets)}
print(f"\n{len(thrust):,} events -> {len(njets):,} hemispheres")
for n in sorted(FRAC):
    print(f"  {n} jet(s): {np.count_nonzero(njets == n):>9,} hemispheres   "
          f"weighted fraction {FRAC[n]:6.1%}")
for s, lab in [(2, "2+3"), (1, "1+4"), (0, "0+5")]:
    cnt = np.count_nonzero(split_min == s)
    if cnt:
        print(f"  {lab} split events: {cnt:,}  ({cnt / len(split_min):.1%})")


## Plot helpers

Fixed color per multiplicity class (never re-assigned when a class is empty),
weighted histograms, percentile-based ranges so odd tails don't set the axes.


In [ ]:
PALETTE = {1: "#2a78d6", 2: "#1baf7a", 3: "#eda100", 4: "#008300"}

def auto_bins(x, nbins=NBINS, qlo=0.1, qhi=99.9, lo=None, hi=None, symmetric=False):
    finite = x[np.isfinite(x)]
    lo = np.percentile(finite, qlo) if lo is None else lo
    hi = np.percentile(finite, qhi) if hi is None else hi
    if symmetric:
        m = max(abs(lo), abs(hi))
        lo, hi = -m, m
    return np.linspace(lo, hi, nbins + 1)

def plot_by_njets(ax, values, bins, xlabel, logy=False):
    for n in (1, 2, 3, 4):
        sel = njets == n
        if not np.any(sel):
            continue
        counts, edges = np.histogram(values[sel], bins=bins, weights=w[sel])
        if DENSITY and counts.sum() > 0:
            counts = counts / (counts.sum() * np.diff(edges))
        ax.stairs(counts, edges, color=PALETTE[n], linewidth=2,
                  label=f"{n} jet{'s' if n > 1 else ''}  ({FRAC.get(n, 0.0):.1%})")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("a.u." if DENSITY else "weighted hemispheres / bin")
    if logy:
        ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=9)
    ax.grid(alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)


## Hemisphere kinematics, split by jet multiplicity

Top rows are the plain four-vector kinematics; then the thrust-projected
components ($|p_{T,\parallel}|$ and $p_{T,\perp}$ are two of the matching
variables), the **partner-hemisphere $\eta$**, and the $\eta$ separation
between a hemisphere and its partner.


In [ ]:
VARIABLES = [
    (hemi["pt"],            r"hemisphere $p_T$  [GeV]",                    dict(lo=0.0),          True),
    (hemi["pz"],            r"hemisphere $p_z$  [GeV]",                    dict(symmetric=True),  True),
    (hemi["energy"],        r"hemisphere energy  [GeV]",                   dict(lo=0.0),          True),
    (hemi["mass"],          r"hemisphere invariant mass  [GeV]",           dict(lo=0.0),          True),
    (hemi["eta"],           r"hemisphere $\eta$",                          dict(symmetric=True),  False),
    (hemi["phi"],           r"hemisphere $\phi$",                          dict(lo=-np.pi, hi=np.pi), False),
    (hemi["abs_pt_par"],    r"$|p_{T,\parallel}|$ along $\hat{n}_T$  [GeV]", dict(lo=0.0),       True),
    (hemi["pt_perp"],       r"$p_{T,\perp}$ w.r.t. $\hat{n}_T$  [GeV]",     dict(symmetric=True), True),
    (hemi["partner_eta"],   r"partner hemisphere $\eta$",                  dict(symmetric=True),  False),
    (hemi["d_eta_partner"], r"$\eta - \eta_{\mathrm{partner}}$",            dict(symmetric=True),  False),
]

ncols = 2
nrows = (len(VARIABLES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(12.5, 3.4 * nrows),
                         constrained_layout=True)
for ax, (vals, label, binkw, logy) in zip(axes.ravel(), VARIABLES):
    plot_by_njets(ax, vals, auto_bins(vals, **binkw), label, logy=logy)
for ax in axes.ravel()[len(VARIABLES):]:
    ax.set_visible(False)
plt.show()


## Transverse thrust, by split type

The mixing method leans on the QCD $2\to2$ dijet approximation: the thrust
axis should split the event into two nearly independent halves. **1+4 splits
are where that approximation is weakest** — compare their thrust spectrum with
the dominant 2+3 splits.


In [ ]:
SPLITS = [(2, "2 + 3 split", "#2a78d6"), (1, "1 + 4 split", "#1baf7a"),
          (0, "0 + 5 split", "#eda100")]

fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.linspace(np.percentile(thrust, 0.1), 1.0, 51)
for s, label, color in SPLITS:
    sel = split_min == s
    if not np.any(sel):
        continue
    counts, edges = np.histogram(thrust[sel], bins=bins, weights=xs_w[sel])
    if DENSITY and counts.sum() > 0:
        counts = counts / (counts.sum() * np.diff(edges))
    ax.stairs(counts, edges, color=color, linewidth=2,
              label=f"{label}  ({np.count_nonzero(sel) / len(sel):.1%} of events)")
ax.set_xlabel("transverse thrust $T$")
ax.set_ylabel("a.u." if DENSITY else "weighted events / bin")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25, linewidth=0.5)
ax.set_axisbelow(True)
plt.show()


### Reading the plots

- **$p_T$, $|p_{T,\parallel}|$**: the two hemispheres of an event balance along
  the thrust axis, so low-multiplicity hemispheres (recoiling against many jets)
  carry comparable $p_T$ but very different mass/energy.
- **mass, energy**: grow strongly with multiplicity — the main reason matching
  requires *identical* jet multiplicity before comparing continuous variables.
- **partner $\eta$ / $\eta - \eta_{\mathrm{partner}}$**: inputs to the
  nearest-neighbor matching (with the $\phi \to -\phi$ reflection); check the
  classes overlap enough for a library to have partners everywhere.
- **$p_{T,\perp}$ shows only two visible curves by construction**: the thrust
  axis maximizes $H(\phi)$, and the stationarity condition $dH/d\phi = 0$ forces
  the two hemispheres of every event to *identical* perpendicular momentum. So
  the 2-jet curve lies exactly under the 3-jet one (and 1-jet under 4-jet) — if
  the pairs ever split apart, the thrust axis is buggy.
- **$\phi$** should be flat — anything else means a detector/selection artifact.
